# 🎙️ MOZHI — Whisper Small Fine-Tuning for Romanized Manglish

**Project:** MOZHI (*Your voice. Your words. Your story.*)  
**Target:** Fine-tuning `openai/whisper-small` (multilingual) on Malayalam-English code-switched speech (MLENSPEECH) to transcribe natural Manglish directly into Romanized Latin text.

### Pipeline Overview:
1. **Environment Setup & GPU Check** (T4 / A100 GPU)
2. **Connect Google Drive** to access MOZHI dataset
3. **Load Speaker-Isolated Dataset Splits** (`train`, `validation`, `test`)
4. **Feature Extraction & Tokenization** (Log-Mel Spectrograms + WhisperTokenizer)
5. **Fine-Tuning with Seq2SeqTrainer** (Mixed precision FP16, Gradient Checkpointing)
6. **Evaluation & Benchmark** (Compare Pretrained Baseline vs Fine-Tuned WER & CER)
7. **Export Checkpoint** for the MOZHI FastAPI Backend

## Step 1: GPU Verification & Dependency Installation

In [ ]:
# Check assigned GPU
!nvidia-smi

In [ ]:
# Install required libraries
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa tensorboard

## Step 2: Mount Google Drive & Set Project Path
Make sure you upload your `MOZHI` folder (or at least `ml/dataset/` and the audio files) to your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Change this path to match where your MOZHI folder is in Google Drive:
MOZHI_ROOT = Path('/content/drive/MyDrive/MOZHI')

assert MOZHI_ROOT.exists(), f"Path does not exist: {MOZHI_ROOT}. Please verify your Google Drive directory!"
print(f"Connected to MOZHI project at: {MOZHI_ROOT}")

## Step 3: Load Speaker-Isolated Dataset Splits
The dataset manifests were created with strict zero-speaker-overlap:
- **Train Set:** SPK001, SPK002, SPK003 (1,856 utterances)
- **Validation Set:** SPK005 (455 utterances)
- **Test Set:** SPK004 (572 utterances)

In [ ]:
import json
from datasets import Dataset, Audio

def load_manifest(manifest_path, base_dir):
    with open(manifest_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()
        if content.startswith('['):
            data = json.loads(content)
        else:
            data = [json.loads(line) for line in content.splitlines() if line.strip()]

    # Resolve audio paths relative to MOZHI_ROOT
    for entry in data:
        p = Path(entry['audio_path'])
        if not p.is_absolute():
            entry['audio_path'] = str((base_dir / p).resolve())
        else:
            entry['audio_path'] = str(p)

    ds = Dataset.from_list(data)
    ds = ds.cast_column('audio_path', Audio(sampling_rate=16000))
    return ds

train_manifest_path = MOZHI_ROOT / 'ml' / 'dataset' / 'train' / 'manifest.json'
val_manifest_path = MOZHI_ROOT / 'ml' / 'dataset' / 'validation' / 'manifest.json'
test_manifest_path = MOZHI_ROOT / 'ml' / 'dataset' / 'test' / 'manifest.json'

train_dataset = load_manifest(train_manifest_path, MOZHI_ROOT)
val_dataset = load_manifest(val_manifest_path, MOZHI_ROOT)
test_dataset = load_manifest(test_manifest_path, MOZHI_ROOT)

print(f"Train samples:      {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples:       {len(test_dataset)}")

# Preview first sample
print("\nSample 0 Preview:")
print("  File ID:   ", train_dataset[0]['file_id'])
print("  Speaker:   ", train_dataset[0]['speaker_id'])
print("  Transcript:", train_dataset[0]['transcript'])

## Step 4: Feature Extraction and Tokenization

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

MODEL_ID = "openai/whisper-small"

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language="Malayalam", task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="Malayalam", task="transcribe")

def prepare_dataset(batch):
    audio = batch["audio_path"]
    # Compute log-Mel spectrogram
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    # Target labels
    batch["labels"] = tokenizer(batch["transcript"]).input_ids
    return batch

print("Preprocessing train and validation sets...")
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names, num_proc=2)
val_dataset = val_dataset.map(prepare_dataset, remove_columns=val_dataset.column_names, num_proc=2)
print("Preprocessing complete!")

## Step 5: Data Collator & Metric Setup

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer_val = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    cer_val = 100 * cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer_val, "cer": cer_val}

## Step 6: Initialize Whisper Model & Training Arguments

In [ ]:
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

# Model settings for fine-tuning
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

OUTPUT_DIR = MOZHI_ROOT / 'models' / 'mozhi-whisper-small'

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=300,
    max_steps=2000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=250,
    eval_steps=250,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

## Step 7: Run Fine-Tuning 🚀

In [ ]:
trainer.train()

# Save final best model and processor
trainer.save_model(str(OUTPUT_DIR))
processor.save_pretrained(str(OUTPUT_DIR))
print(f"\nFine-tuned MOZHI model saved successfully to: {OUTPUT_DIR}")

## Step 8: Rigorous Evaluation — Baseline vs Fine-Tuned (Held-Out Test Set: SPK004)
We evaluate both models on the held-out test split (Speaker SPK004, which neither model was trained on).

In [ ]:
from jiwer import wer, cer
from tqdm import tqdm
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load Baseline Model
print("Loading original baseline Whisper Small...")
baseline_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device)
baseline_model.eval()

# 2. Load Fine-Tuned Model
print("Loading fine-tuned MOZHI model...")
finetuned_model = WhisperForConditionalGeneration.from_pretrained(str(OUTPUT_DIR)).to(device)
finetuned_model.eval()

references = []
baseline_preds = []
finetuned_preds = []

eval_samples = min(100, len(test_dataset))  # evaluate first 100 test samples
print(f"Evaluating {eval_samples} samples from held-out test split (Speaker SPK004)...")

for i in tqdm(range(eval_samples)):
    sample = test_dataset[i]
    audio = sample['audio_path']['array']
    ref = sample['transcript']
    references.append(ref)

    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").to(device)

    with torch.no_grad():
        # Baseline prediction
        b_forced = processor.get_decoder_prompt_ids(language="english", task="transcribe")
        b_ids = baseline_model.generate(inputs.input_features, forced_decoder_ids=b_forced, max_new_tokens=128)
        b_text = processor.batch_decode(b_ids, skip_special_tokens=True)[0].strip().lower()
        baseline_preds.append(b_text)

        # Fine-tuned prediction
        ft_forced = processor.get_decoder_prompt_ids(language="malayalam", task="transcribe")
        ft_ids = finetuned_model.generate(inputs.input_features, forced_decoder_ids=ft_forced, max_new_tokens=128)
        ft_text = processor.batch_decode(ft_ids, skip_special_tokens=True)[0].strip().lower()
        finetuned_preds.append(ft_text)

# Compute metrics
b_wer = wer(references, baseline_preds) * 100
b_cer = cer(references, baseline_preds) * 100
ft_wer = wer(references, finetuned_preds) * 100
ft_cer = cer(references, finetuned_preds) * 100

print("\n" + "="*60)
print("FINAL EVALUATION REPORT — HELD-OUT TEST SET (SPK004)")
print("="*60)
print(f"Original Whisper Baseline : WER = {b_wer:.2f}% | CER = {b_cer:.2f}%")
print(f"Fine-Tuned MOZHI Model    : WER = {ft_wer:.2f}% | CER = {ft_cer:.2f}%")
print(f"WER Improvement           : {b_wer - ft_wer:+.2f}%")
print(f"CER Improvement           : {b_cer - ft_cer:+.2f}%")
print("="*60)

# Comparison DataFrame
df = pd.DataFrame({
    "Reference (Target)": references[:10],
    "Baseline Output": baseline_preds[:10],
    "Fine-Tuned Output": finetuned_preds[:10]
})
df